## Broadcast Joins & Data Skew

Joins are the most expensive operation in distributed Spark jobs — by default both sides must be shuffled across the network so matching keys land on the same executor. Broadcast joins eliminate that shuffle entirely by sending a copy of the smaller table to every executor. Data skew is the other join enemy: when a few keys dominate the data, some tasks receive far more rows than others and stall the entire stage.

![Broadcast Join vs Sort-Merge Join](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-broadcast-join.svg)

In this notebook you will learn:
- How the **broadcast hash join** works and when Spark chooses it automatically
- How to force a broadcast with the `broadcast()` hint
- How to read join strategies in `explain()` output
- What data skew is, how to detect it, and three techniques to fix it
- How AQE handles skew automatically at runtime

### Join Strategies in Spark

Spark chooses from several join algorithms depending on data size and available hints:

| Strategy | Trigger | Shuffle | Best for |
|---|---|---|---|
| **Broadcast Hash Join** | One side ≤ `autoBroadcastJoinThreshold` (default 10 MB) or `broadcast()` hint | None | Small dimension tables |
| **Sort-Merge Join** | Both sides large, no broadcast possible | Both sides | Large fact-to-fact joins |
| **Shuffle Hash Join** | One side fits in memory per partition | One side | Medium tables |
| **Cartesian Join** | `CROSS JOIN` or no join condition | Full cross product | Explicit cross-joins only |

The broadcast hash join is by far the most impactful optimisation available for typical star-schema queries — joining a large fact table with small dimension tables.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, count, sum as _sum, avg, broadcast, spark_partition_id

spark = (
    SparkSession.builder
    .appName("BroadcastJoinsDataSkew")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

DATA = os.path.join(os.path.dirname(os.path.abspath("14-broadcast-joins-data-skew.ipynb")), "data")

customers = spark.read.option("header", "true").schema(StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("date_of_birth", DateType(),      True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
])).csv(f"{DATA}/customers")

card_accounts = spark.read.schema(StructType([
    StructField("card_id",      StringType(),      False),
    StructField("customer_id",  StringType(),      False),
    StructField("card_type",    StringType(),      False),
    StructField("network",      StringType(),      True),
    StructField("credit_limit", DecimalType(18,2), True),
    StructField("issued_on",    DateType(),        False),
    StructField("expiry_date",  DateType(),        False),
    StructField("status",       StringType(),      False),
])).json(f"{DATA}/card_accounts")

card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),      False),
    StructField("card_id",     StringType(),      False),
    StructField("customer_id", StringType(),      False),
    StructField("amount",      DecimalType(18,2), False),
    StructField("merchant",    StringType(),      True),
    StructField("category",    StringType(),      True),
    StructField("txn_ts",      TimestampType(),   False),
    StructField("status",      StringType(),      False),
    StructField("is_fraud",    BooleanType(),     False),
])).parquet(f"{DATA}/card_transactions")

loan_accounts = spark.read.option("multiLine", "true").schema(StructType([
    StructField("loan_id",       StringType(),      False),
    StructField("customer_id",   StringType(),      False),
    StructField("loan_type",     StringType(),      False),
    StructField("principal",     DecimalType(18,2), False),
    StructField("interest_rate", DoubleType(),      False),
    StructField("tenure_months", IntegerType(),     False),
    StructField("disbursed_on",  DateType(),        False),
    StructField("status",        StringType(),      False),
])).json(f"{DATA}/loan_accounts")

payments = spark.read.format("delta").load(f"{DATA}/payments")

print(f"customers     : {customers.count():>6} rows")
print(f"card_accounts : {card_accounts.count():>6} rows")
print(f"card_txns     : {card_txns.count():>6} rows")
print(f"loan_accounts : {loan_accounts.count():>6} rows")
print(f"payments      : {payments.count():>6} rows")

### Auto-Broadcast: Reading the Default Threshold

`spark.sql.autoBroadcastJoinThreshold` (default **10 MB**) controls the size below which Spark automatically broadcasts a table. Spark estimates table sizes from file metadata and statistics — if the estimate is below the threshold, it chooses a broadcast hash join without any hint from you.

In [ ]:
print("autoBroadcastJoinThreshold:",
      spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Join card_txns (large) with customers (small)
# Spark may auto-broadcast customers if its size estimate is below the threshold
auto_join = card_txns.join(customers, on="customer_id", how="inner")

print("\n=== AUTO JOIN PLAN ===")
auto_join.explain()

### Forcing a Broadcast with the broadcast() Hint

When Spark's size estimate is inaccurate (common after transformations that change row counts), or when you want to be explicit, wrap the small DataFrame with `broadcast()`. Spark will always use a broadcast hash join regardless of estimated size.

**Rule**: broadcast the *smaller* side. Broadcasting a large table exhausts executor memory and causes OOM errors.

In [ ]:
from pyspark.sql.functions import broadcast

# Explicit broadcast — customers is small, always safe to broadcast
broadcast_join = card_txns.join(broadcast(customers), on="customer_id", how="inner")

print("=== BROADCAST JOIN PLAN ===")
broadcast_join.explain()
# Look for BroadcastHashJoin in the plan — no Exchange operators on the customers side

In [ ]:
# Broadcast card_accounts (small dimension) when joining with card_txns
txn_enriched = (
    card_txns
    .join(broadcast(customers),     on="customer_id", how="left")
    .join(broadcast(card_accounts), on="card_id",     how="left")
    .select(
        "txn_id", "customer_id", col("full_name"),
        col("city"), "amount", "category",
        col("card_type"), col("network"), "status",
    )
)

print("=== DUAL BROADCAST PLAN ===")
txn_enriched.explain()
# Both customers and card_accounts are broadcast — no shuffle on either dimension side

### broadcast() Hint in SQL

The `/*+ BROADCAST(table_alias) */` hint applies the same strategy directly inside a SQL query.

In [ ]:
card_txns.createOrReplaceTempView("card_transactions")
customers.createOrReplaceTempView("customers")
card_accounts.createOrReplaceTempView("card_accounts")
loan_accounts.createOrReplaceTempView("loan_accounts")
payments.createOrReplaceTempView("payments")

result = spark.sql("""
    SELECT /*+ BROADCAST(c) */
           t.txn_id,
           t.customer_id,
           c.full_name,
           c.city,
           t.amount,
           t.category
    FROM   card_transactions t
    JOIN   customers c USING (customer_id)
    WHERE  t.status = 'SUCCESS'
    LIMIT  10
""")
result.show(truncate=False)
result.explain()

### Disabling Auto-Broadcast to See the Difference

Setting `autoBroadcastJoinThreshold` to `-1` disables automatic broadcasting. The same join then produces a sort-merge join with two Exchange (shuffle) operators — one per side.

In [ ]:
# Disable auto-broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

no_broadcast = card_txns.join(customers, on="customer_id", how="inner")
print("=== WITHOUT BROADCAST (sort-merge join) ===")
no_broadcast.explain()
# Now you will see Exchange operators on both sides — a full sort-merge join

# Re-enable with a generous threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10 MB
print("\nautoBroadcastJoinThreshold restored to 10 MB")

### When NOT to Broadcast

Broadcasting is only safe when the table fits in executor memory. A useful guideline:

| Table size | Recommendation |
|---|---|
| < 10 MB | Auto-broadcast (Spark default) |
| 10 MB – 100 MB | Force broadcast with hint — safe on most clusters |
| 100 MB – 500 MB | Broadcast with caution — test on your cluster's executor memory |
| > 500 MB | Do not broadcast — use sort-merge join or AQE |

Broadcasting `loan_accounts` against `card_txns` is safe (loan_accounts is small). Broadcasting `card_txns` against `loan_accounts` would be wrong — `card_txns` is the large table and would exhaust executor memory.

In [ ]:
# Correct: broadcast the small side
good = card_txns.join(broadcast(loan_accounts), on="customer_id", how="left")
good.explain()  # BroadcastHashJoin — loan_accounts broadcast, card_txns stays distributed

# Incorrect pattern to avoid: broadcast(card_txns) — the large table
# bad = loan_accounts.join(broadcast(card_txns), on="customer_id")  # OOM risk!
print("Always broadcast the smaller side of the join.")

### Data Skew — What It Is

Data skew occurs when the distribution of rows across partitions is highly uneven — some partitions have far more rows than others. Because a stage cannot complete until the last task finishes, one large partition stalls the entire stage regardless of how fast all other tasks run.

In banking data, skew is common on `customer_id` joins: a handful of high-volume customers or merchants generate far more transactions than average. When Spark shuffles by `customer_id`, the partitions for those hot keys receive a disproportionate number of rows.

**Symptoms of skew:**
- One task takes 10–100× longer than the median task in a stage (visible in Spark UI → Stages)
- One executor shows high GC pressure or spills to disk while others are idle
- The stage completes quickly except for the last one or two tasks

### Detecting Skew — Partition Size Inspection

In [ ]:
from pyspark.sql.functions import spark_partition_id, count, col

# Check row distribution across partitions of card_txns after a shuffle
shuffled = card_txns.repartition(8, col("customer_id"))  # hash-partition by customer

partition_sizes = (
    shuffled
    .withColumn("pid", spark_partition_id())
    .groupBy("pid")
    .agg(count("txn_id").alias("row_count"))
    .orderBy(col("row_count").desc())
)
partition_sizes.show()
# Uneven row_count across partitions signals skew

In [ ]:
# Identify the hot keys — which customers generate the most transactions?
from pyspark.sql.functions import count, col

txn_per_customer = (
    card_txns
    .groupBy("customer_id")
    .agg(count("txn_id").alias("txn_count"))
    .orderBy(col("txn_count").desc())
)
txn_per_customer.show(10)
# The top customers are the hot keys that will dominate a customer_id shuffle

### Fix 1 — Salting: Manually Breaking Hot Keys

Salting appends a random integer suffix to the join key on the large side, splitting one hot key into multiple artificial sub-keys. The small side is exploded to produce one row per salt value so that sub-key rows can still match.

This is the most reliable manual fix for severe skew when AQE is not available or not sufficient.

```
Large side key:  "CUST0001"  →  "CUST0001_0", "CUST0001_1", "CUST0001_2"
Small side key:  "CUST0001"  →  "CUST0001_0", "CUST0001_1", "CUST0001_2"  (exploded)
```

In [ ]:
from pyspark.sql.functions import (
    col, concat, lit, floor, rand, array, explode, monotonically_increasing_id
)

SALT_BUCKETS = 4  # split each hot key across 4 sub-partitions

# Step 1: add a random salt to the large side (card_txns)
txns_salted = card_txns.withColumn(
    "salted_key",
    concat(col("customer_id"), lit("_"), (floor(rand() * SALT_BUCKETS)).cast("int"))
)

# Step 2: explode the small side (customers) to produce one row per salt value
customers_exploded = (
    customers
    .withColumn("salt_array", array([lit(i) for i in range(SALT_BUCKETS)]))
    .withColumn("salt", explode(col("salt_array")))
    .withColumn(
        "salted_key",
        concat(col("customer_id"), lit("_"), col("salt").cast("string"))
    )
    .drop("salt_array", "salt")
)

print(f"customers rows before explode : {customers.count()}")
print(f"customers rows after explode  : {customers_exploded.count()} (×{SALT_BUCKETS})")

# Step 3: join on the salted key
salted_result = (
    txns_salted
    .join(broadcast(customers_exploded), on="salted_key", how="left")
    .drop("salted_key")
    .select(
        "txn_id", "customer_id", col("full_name"), col("city"), "amount", "status"
    )
)

print(f"\nSalted join result: {salted_result.count()} rows")
salted_result.show(5, truncate=False)

### Fix 2 — AQE Skew Join Handling

Adaptive Query Execution detects skewed partitions at runtime and automatically splits them into sub-tasks. No code change is needed — enable AQE and configure the skew thresholds.

AQE considers a partition skewed when:
- Its size is ≥ `spark.sql.adaptive.skewJoin.skewedPartitionFactor` × median partition size, **and**
- Its size is ≥ `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (default 256 MB)

In [ ]:
# Check AQE skew settings
for key in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
]:
    try:
        print(f"  {key.split('.')[-1]:<45}: {spark.conf.get(key)}")
    except Exception:
        print(f"  {key.split('.')[-1]:<45}: (not set — using default)")

In [ ]:
# Enable AQE (default since Spark 3.2) and tune skew thresholds
spark.conf.set("spark.sql.adaptive.enabled",              "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",     "true")
# Lower the threshold so AQE triggers on our small demo dataset
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")

# With AQE, this join benefits from automatic skew handling — no salting needed
aqe_join = (
    card_txns
    .join(customers, on="customer_id", how="inner")
    .groupBy("city")
    .agg(_sum("amount").alias("total_spend"))
    .orderBy(col("total_spend").desc())
)
aqe_join.show()
print("AQE skew join complete — skewed partitions split automatically at runtime")

### Fix 3 — repartitionByRange Before a Join

`repartitionByRange(n, col)` sorts rows by a column and divides them into equal-sized ranges, producing balanced partitions regardless of key frequency. Unlike hash repartition, range repartition distributes based on value ranges — hot keys are split across multiple partitions.

Use this as a pre-join step when AQE is unavailable or when skew is severe enough to need explicit control.

In [ ]:
from pyspark.sql.functions import count, col, spark_partition_id

# Pre-balance card_txns by customer_id range before a join
txns_balanced = card_txns.repartitionByRange(8, col("customer_id"))

# Verify balance — row counts should be more even than hash repartition
(
    txns_balanced
    .withColumn("pid", spark_partition_id())
    .groupBy("pid")
    .agg(count("txn_id").alias("row_count"))
    .orderBy("pid")
).show()

# Now join — balanced input means balanced join tasks
balanced_join = (
    txns_balanced
    .join(broadcast(customers), on="customer_id", how="left")
    .select("txn_id", "customer_id", col("city"), "amount", "status")
)
print(f"Balanced join result: {balanced_join.count()} rows")

### Skew in GroupBy Aggregations

Skew is not limited to joins — `groupBy` on a skewed key has the same problem. A small number of high-frequency keys dominate one or a few output partitions.

The fix for aggregation skew is a **two-phase aggregation** (map-side partial aggregate then reduce), which `groupBy` + `agg` already does internally. For extreme cases, manually pre-aggregate with a salted key and then aggregate again.

In [ ]:
from pyspark.sql.functions import sum as _sum, count, floor, rand, concat, lit, col

SALT = 4  # number of partial buckets

# Phase 1: partial aggregation with salted key — reduces shuffle data volume
partial = (
    card_txns
    .withColumn("salt", (floor(rand() * SALT)).cast("int"))
    .withColumn("salted_key", concat(col("customer_id"), lit("_"), col("salt")))
    .groupBy("salted_key", "customer_id")
    .agg(
        count("txn_id").alias("partial_count"),
        _sum("amount").alias("partial_sum"),
    )
)

# Phase 2: final aggregation — sum the partial results per true customer_id
final = (
    partial
    .groupBy("customer_id")
    .agg(
        _sum("partial_count").alias("total_txns"),
        _sum("partial_sum").alias("total_spend"),
    )
    .orderBy(col("total_spend").desc())
)
final.show(10)

### Summary

**Broadcast hash join** sends the smaller table to every executor so the join runs locally with no shuffle. It is the most impactful join optimisation for star-schema queries.

**Auto-broadcast**: Spark broadcasts automatically when a table's size estimate is below `spark.sql.autoBroadcastJoinThreshold` (default 10 MB).

**`broadcast()` hint**: wrap the small DataFrame with `broadcast()` to force a broadcast hash join. In SQL use `/*+ BROADCAST(alias) */`.

**Do not broadcast large tables** — broadcasting a table that does not fit in executor memory causes OOM errors. Guideline: safe below 100 MB, risky above 500 MB.

**Data skew** occurs when a few keys dominate the partition distribution. One slow task stalls the entire stage.

Three fixes:
1. **Salting** — append a random suffix to hot keys on the large side; explode the small side to match. Works without AQE.
2. **AQE skew join** — enable `spark.sql.adaptive.skewJoin.enabled`; AQE detects and splits skewed partitions automatically at runtime.
3. **`repartitionByRange`** — pre-balance by value range before a join or aggregation for explicit control.

**Prefer AQE** for skew handling in modern Spark (3.2+) — it requires no code change and adapts to actual runtime data distributions.